In [ ]:
import numpy as np
np.set_printoptions(legacy='1.25')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

data = '/content/drive/MyDrive/WE WILL WIN ISEF/Data/FinalData.csv'
df = pd.read_csv(data)

df = df.T

df.head(60)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

df_cleaned = df.dropna(subset=[df.columns[-1]])

X = df_cleaned.iloc[:, :-1]
y = df_cleaned.iloc[:, -1].values.astype(int)

print(f"Cleaned data: {X.shape[0]} samples remaining.")
print(f"Labels: {np.unique(y, return_counts=True)}")

gene_mapping = df.loc['Gene_Symbol'].values[1:]

X.columns = gene_mapping

X.columns = X.columns.astype(str)

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    class_weight='balanced',
    max_features='sqrt'
)

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

cv_results = cross_validate(
    rf, X_scaled, y,
    cv=rkf,
    scoring=['accuracy', 'roc_auc'],
    n_jobs=-1
)

acc_mean = cv_results['test_accuracy'].mean()
acc_std = cv_results['test_accuracy'].std()
auc_mean = cv_results['test_roc_auc'].mean()
auc_std = cv_results['test_roc_auc'].std()

print("--- Model Performance (Repeated Stratified 5-Fold) ---")
print(f"Mean Accuracy: {acc_mean:.2%} (+/- {acc_std:.2%})")
print(f"Mean AUC:      {auc_mean:.3f} (+/- {auc_std:.3f})")

rf.fit(X_scaled, y)
importance_df = pd.DataFrame({
    'Gene_Symbol': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.head(50).to_csv('Top50_features.csv', index=False)

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

base_svc = SVC(kernel="linear", C=0.1)

selector = RFE(estimator=base_svc, n_features_to_select=30, step=100)

bagging_model = BaggingClassifier(estimator=selector, n_estimators=10, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', bagging_model)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = cross_validate(pipeline, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])

print(f"Mean Accuracy: {np.mean(results['test_accuracy']):.2f}")
print(f"Mean AUC-ROC: {np.mean(results['test_roc_auc']):.2f}")

import pandas as pd
import numpy as np

pipeline.fit(X, y)
sub_rfe_estimators = pipeline.named_steps['classifier'].estimators_

feature_votes = np.array([rfe_est.support_ for rfe_est in sub_rfe_estimators])

vote_counts = feature_votes.sum(axis=0)

gene_importance = pd.DataFrame({
    'Gene': X.columns,
    'Votes': vote_counts
}).sort_values(by='Votes', ascending=False)

top_genes = gene_importance[gene_importance['Votes'] > 5]
print("Genes consistently selected across the ensemble:")
print(top_genes.head(30))

import seaborn as sns
import matplotlib.pyplot as plt

top_gene_names = gene_importance[gene_importance['Votes'] >= 7]['Gene'].tolist()
X_top = X[top_gene_names]

y_colors = pd.Series(y).map({0: "skyblue", 1: "salmon"})
y_colors.index = X.index

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

if not top_gene_names:
    print("Error: No genes were found with high votes. Try lowering the vote threshold (e.g., >= 5).")
else:
    X_numeric = X[top_gene_names].apply(pd.to_numeric, errors='coerce')

    X_numeric = X_numeric.dropna(axis=1, how='all')

    X_numeric = X_numeric.fillna(X_numeric.median())

    if X_numeric.empty:
        print("X_numeric is still empty. Check if your column names in 'X' match 'top_gene_names'.")
    else:
        g = sns.clustermap(X_numeric.T,
                           cmap="vlag",
                           standard_scale=0,
                           col_colors=y_colors,
                           figsize=(12, 10),
                           method='ward')
        plt.show()

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

top_20_genes = gene_importance.head(20)['Gene'].tolist()

X_20 = X[top_20_genes].apply(pd.to_numeric, errors='coerce')
X_20 = X_20.fillna(X_20.median())

sample_colors = pd.Series(y).map({0: "skyblue", 1: "salmon"})
sample_colors.index = X.index

g = sns.clustermap(X_20.T,
                   cmap="vlag",
                   standard_scale=0,
                   col_colors=sample_colors,
                   figsize=(11, 9.42),
                   method='ward',
                   metric='euclidean')

plt.title("Signature of Top 20 Genes: RRMS (Blue) vs SPMS (Red)")
plt.show()

modules = {
    'ribosomal_score': ['RPS11', 'RPS12', 'RPS17', 'EIF3E', 'EIF2AK1'],
    'immune_signaling': ['TYROBP', 'SNX27', 'SNX3', 'LYL1'],
    'membrane_transport': ['AQP9', 'TMEM123', 'TMEM49', 'SNX27']
}

X_pathways = pd.DataFrame()
for name, genes in modules.items():
    existing_genes = [g for g in genes if g in X.columns]
    X_pathways[name] = X[existing_genes].mean(axis=1)

X_pathways['LOC441087'] = X['LOC441087']

from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

svm_pathway = SVC(kernel='linear', C=1.0)
scores = cross_val_score(svm_pathway, X_pathways, y, cv=5)

print(f"Pathway-Based Accuracy: {scores.mean():.2f} (+/- {scores.std():.2f})")

import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

top_10_genes = gene_importance.head(10)['Gene'].tolist()
X_top = X[top_10_genes].copy()

ratios_df = pd.DataFrame(index=X.index)
for g1, g2 in combinations(top_10_genes, 2):
    ratios_df[f"{g1}_{g2}_ratio"] = X_top[g1] / (X_top[g2] + 1e-9)

ratio_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=0.1, probability=True))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ratio_scores = cross_val_score(ratio_pipeline, ratios_df, y, cv=cv)

print(f"Ratio-Based SVM Accuracy: {ratio_scores.mean():.2f}")

from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

ratio_pipeline.fit(ratios_df, y)
RocCurveDisplay.from_estimator(ratio_pipeline, ratios_df, y)
plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curve for Ratio-Based SVM")
plt.show()

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, auc

y_probs = cross_val_predict(ratio_pipeline, ratios_df, y, cv=cv, method='predict_proba')[:, 1]

fpr, tpr, thresholds = roc_curve(y, y_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f'CV ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate (RRMS misclassified as SPMS)')
plt.ylabel('True Positive Rate (SPMS correctly identified)')
plt.title('Cross-Validated ROC Curve')
plt.legend()
plt.show()

import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.base import BaseEstimator, TransformerMixin

class LogRatioTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, gene_names):
        self.gene_names = gene_names

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.gene_names)
        ratios = {}

        X_safe = X_df + 1e-6
        X_log = np.log2(X_safe)

        for g1, g2 in combinations(self.gene_names, 2):
            ratios[f"{g1}_{g2}_logratio"] = X_log[g1] - X_log[g2]

        return pd.DataFrame(ratios)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, cross_val_score, permutation_test_score

top_10 = gene_importance.head(10)['Gene'].tolist()
X_small = X[top_10]

selector_model = LinearSVC(
    penalty="l1",
    dual=False,
    C=1.0,
    max_iter=20000,
    random_state=42
)

robust_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('ratios', LogRatioTransformer(gene_names=top_10)),
    ('scaler', StandardScaler()),
    ('selector', SelectFromModel(selector_model)),
    ('svm', SVC(kernel='linear', probability=True, max_iter=20000))
])

param_grid = {'svm__C': [0.01, 0.1, 1, 10]}
inner_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
grid_search = GridSearchCV(estimator=robust_pipe, param_grid=param_grid, cv=inner_cv, scoring='roc_auc')

outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
nested_scores = cross_val_score(grid_search, X_small, y, cv=outer_cv)

print(f"--- Final Pipeline Performance ---")
print(f"Unbiased Mean Accuracy: {nested_scores.mean():.2%} (+/- {nested_scores.std():.2%})")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

grid_search.fit(X_small, y)

y_probs = grid_search.predict_proba(X_small)[:, 1]

thresholds = np.linspace(0, 1, 100)
accuracies = [np.mean((y_probs >= t) == y) for t in thresholds]
f1_scores = [f1_score(y, (y_probs >= t).astype(int)) for t in thresholds]

best_f1_threshold = thresholds[np.argmax(f1_scores)]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, accuracies, label='Overall Accuracy', lw=2, color='steelblue')
plt.plot(thresholds, f1_scores, label='F1-Score (Balance)', lw=2, color='darkorange')
plt.axvline(best_f1_threshold, color='red', linestyle='--',
            label=f'Optimal Threshold: {best_f1_threshold:.2f}')

plt.title("Threshold Optimization: Finding the 'Sweet Spot'", fontsize=14)
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

grid_search.fit(X_small, y)

probs = grid_search.predict_proba(X_small)[:, 1]
final_preds = (probs >= 0.40).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y, final_preds, target_names=['RRMS', 'SPMS']))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

data = {
    'Class': ['RRMS', 'SPMS'],
    'Precision': [0.86, 0.71],
    'Recall': [0.84, 0.75],
    'F1-Score': [0.85, 0.73]
}
df_metrics = pd.DataFrame(data).set_index('Class')

cm = np.array([[30, 6],
               [5, 15]])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted RRMS', 'Predicted SPMS'],
            yticklabels=['Actual RRMS', 'Actual SPMS'], ax=ax[0])
ax[0].set_title('Confusion Matrix', fontsize=14, pad=15)

df_metrics.plot(kind='bar', ax=ax[1], color=['#4e79a7', '#f28e2b', '#e15759'])
ax[1].set_title('Performance Metrics by Class', fontsize=14, pad=15)
ax[1].set_ylim(0, 1.0)
ax[1].set_ylabel('Score')
ax[1].legend(loc='lower right')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc, confusion_matrix

fig, ax = plt.subplots(figsize=(8, 8))
tprs = []
mean_fpr = np.linspace(0, 1, 100)
cv = StratifiedKFold(n_splits=5)

threshold_fprs = []
threshold_tprs = []
CUSTOM_THRESHOLD = 0.40

for i, (train_index, test_index) in enumerate(cv.split(X_small, y)):
    X_train, X_test = X_small.iloc[train_index], X_small.iloc[test_index]
    y_train = y.iloc[train_index] if hasattr(y, 'iloc') else y[train_index]
    y_test = y.iloc[test_index] if hasattr(y, 'iloc') else y[test_index]

    robust_pipe.fit(X_train, y_train)

    y_probs_fold = robust_pipe.predict_proba(X_test)[:, 1]
    fpr_f, tpr_f, _ = roc_curve(y_test, y_probs_fold)

    interp_tpr = np.interp(mean_fpr, fpr_f, tpr_f)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)

    y_pred_custom = (y_probs_fold >= CUSTOM_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    threshold_tprs.append(tp / (tp + fn))
    threshold_fprs.append(fp / (fp + tn))

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)

ax.plot(mean_fpr, mean_tpr, color="royalblue",
        label=f"Mean ROC (AUC = {mean_auc:.2f})", lw=4)
ax.plot([0, 1], [0, 1], linestyle="--", lw=2, color="salmon", label="Random Chance")

mean_th_fpr = np.mean(threshold_fprs)
mean_th_tpr = np.mean(threshold_tprs)

ax.set(title="Mean Diagnostic Performance (ROC)",
       xlabel="False Positive Rate (1 - Specificity)",
       ylabel="True Positive Rate (Sensitivity)")
ax.legend(loc="lower right")
plt.show()

clean_params = {k.split('__')[1]: v for k, v in grid_search.best_params_.items()}

fast_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, **clean_params))
])

optimized_model_fast = ThresholdClassifier(fast_pipe, threshold=0.39)

print("Running optimized permutation test...")
score, perm_scores, pvalue = permutation_test_score(
    optimized_model_fast, X_small, y, cv=outer_cv, n_permutations=500, n_jobs=-1, verbose=3
)

print(f"Permutation P-value: {pvalue:.4f}")

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.multiclass import check_classification_targets
from sklearn.utils.validation import check_is_fitted
import numpy as np

class ThresholdClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, threshold=0.4):
        self.estimator = estimator
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        check_classification_targets(y)
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

optimized_model_fast = ThresholdClassifier(fast_pipe, threshold=0.40)

print("Running optimized permutation test (AUC)...")
score, perm_scores, pvalue = permutation_test_score(
    optimized_model_fast,
    X_small,
    y,
    cv=outer_cv,
    n_permutations=500,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

print(f"\nActual Model AUC: {score:.4f}")
print(f"Permutation P-value: {pvalue:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(perm_scores, bins=20, color='lightgray', edgecolor='white', label='Null Distribution (Random)')
plt.axvline(score, color='red', linestyle='--', lw=2, label=f'Actual Model Score ({score:.2f})')
plt.title(f'Permutation Test (p-value = {pvalue:.4f})', fontsize=14)
plt.xlabel('Accuracy Score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

import matplotlib.pyplot as plt
import seaborn as sns

def plot_p_value(perm_scores, observed_score, p_value):
    plt.figure(figsize=(10, 6))

    sns.histplot(perm_scores, bins=30, kde=True, color='slategray', alpha=0.6, label='Null Distribution (Random)')

    plt.axvline(observed_score, color='crimson', linestyle='--', lw=3, label='Our Model Performance')

    plt.text(0.05, 0.95, f'p-value = {p_value:.4f}',
             transform=plt.gca().transAxes, fontsize=14, fontweight='bold',
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round'))

    plt.title("Statistical Significance Test (500 Permutations)", fontsize=16, pad=15)
    plt.xlabel("Model Metric (AUC)", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(axis='y', alpha=0.3)

    sns.despine()

    plt.tight_layout()
    plt.show()

plot_p_value(perm_scores, score, pvalue)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

models = {
    'Random Forest': RandomForestClassifier(n_estimators=1000, class_weight='balanced', random_state=42),
    'Linear SVM': SVC(kernel='linear', class_weight='balanced', probability=True),
    'Logistic Reg': LogisticRegression(max_iter=1000, class_weight='balanced')
}

print("--- Model Comparison (5-Fold CV) ---")
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f"{name} Accuracy: {scores.mean():.2%} (+/- {scores.std():.2%})")

from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

svm_model = SVC(kernel='linear', class_weight='balanced', C=0.1, probability=True)

cv_strategy = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

svm_acc = cross_val_score(svm_model, X_scaled, y, cv=cv_strategy, scoring='accuracy')

svm_auc = cross_val_score(svm_model, X_scaled, y, cv=cv_strategy, scoring='roc_auc')

print(f"Linear SVM Mean Accuracy: {svm_acc.mean():.2%} (+/- {svm_acc.std():.2%})")
print(f"Linear SVM Mean AUC-ROC:  {svm_auc.mean():.4f} (+/- {svm_auc.std():.4f})")

!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from sklearn.model_selection import cross_val_score

samplers = {
    'Standard (None)': None,
    'SMOTE': SMOTE(random_state=42, k_neighbors=3),
    'ADASYN': ADASYN(random_state=42, n_neighbors=3),
    'Random OverSample': RandomOverSampler(random_state=42)
}

results = {}

for name, sampler in samplers.items():
    if sampler:
        test_pipe = ImbPipeline([
            ('sampler', sampler),
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, **clean_params))
        ])
    else:
        test_pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, **clean_params))
        ])

    final_model = ThresholdClassifier(test_pipe, threshold=0.40)

    score = cross_val_score(final_model, X_small, y, cv=outer_cv, scoring='accuracy').mean()
    results[name] = score

for name, score in results.items():
    print(f"{name}: {score:.2%}")
